# Multiclass Dataset 실습

이 노트북은 `MulticlassDataset`의 이미지 정규화, one-hot 인코딩, `Dataloader` 배치 생성 동작을 직접 실행하고 검증하는 실습 자료이다.
이전 노트북의 변수나 실행 결과를 사용하지 않으며, 이 노트북 단독으로 완전히 실행할 수 있다.

**실행 환경**: `conda run -n numpy_py311` (GPU 불필요)

**데이터셋 경로**: `/mnt/d/datasets/mnist/` (train/test `.gz` 4개 파일 필요)

**목표**
- `normalize`와 `to_flat`이 이미지를 `(N, 784)` float32로 변환하는 과정을 확인한다.
- `one_hot`이 정수 레이블을 `(N, 10)` one-hot 벡터로 변환하는 과정을 확인한다.
- `MulticlassDataset`이 transform을 eager하게 적용하여 `__getitem__`에서 변환 없이 슬라이싱만 수행함을 확인한다.
- `Dataloader`로 배치를 생성하고 학습 루프에서 사용하는 방식을 시연한다.

## 0. 환경 설정

In [ ]:
# sys.path setup -- excluded from jupyter book build
import os
import sys

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"project_root={PROJECT_ROOT}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from src.data.mnist import load_images, load_labels
from src.data import transforms as T
from src.data.datasets import MulticlassDataset
from src.data.dataloader import Dataloader

## 1. 개요

`MulticlassDataset`은 10개 클래스 중 하나를 예측하는 multiclass classification task용 Dataset이다.
생성자에서 이미지에 `normalize + to_flat`, 레이블에 `one_hot`을 eager하게 적용하여 모든 변환을 미리 완료한다.
`__getitem__`은 변환된 배열을 인덱싱만 하므로 배치 생성 시 추가 연산이 없다.

각 변환 함수의 역할은 다음과 같다.

| 함수 | 입력 | 출력 | 역할 |
|---|---|---|---|
| `normalize` | `(N, 28, 28) uint8` | `(N, 28, 28) float32` | `/255.0` — 픽셀 값을 `[0, 1]`로 정규화 |
| `to_flat` | `(N, 28, 28) float32` | `(N, 784) float32` | `reshape` — MLP 입력 형태로 펼침 |
| `one_hot` | `(N,) uint8` | `(N, 10) float32` | 정수 레이블을 길이 10 one-hot 벡터로 변환 |

## 2. 이미지 변환 확인

`load_images`가 반환한 원본 `uint8` 배열에 `normalize`와 `to_flat`을 순서대로 적용한다.
각 단계에서 shape, dtype, 값 범위가 어떻게 바뀌는지 확인한다.

In [ ]:
images_raw = load_images("train")   # (60000, 28, 28) uint8

images_norm = T.normalize(images_raw)          # (60000, 28, 28) float32
images_flat = T.to_flat(images_norm)           # (60000, 784) float32

print(f"raw   : shape={images_raw.shape},  dtype={images_raw.dtype},  "
      f"min={images_raw.min()},  max={images_raw.max()}")
print(f"norm  : shape={images_norm.shape}, dtype={images_norm.dtype}, "
      f"min={images_norm.min():.4f}, max={images_norm.max():.4f}")
print(f"flat  : shape={images_flat.shape},  dtype={images_flat.dtype}, "
      f"min={images_flat.min():.4f}, max={images_flat.max():.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3))

# raw image
axes[0].imshow(images_raw[0], cmap='gray', vmin=0, vmax=255)
axes[0].set_title(f"raw: uint8 [0, 255]")
axes[0].axis('off')

# normalized image
axes[1].imshow(images_norm[0], cmap='gray', vmin=0, vmax=1)
axes[1].set_title(f"normalized: float32 [0, 1]")
axes[1].axis('off')

# flattened — reshape back for display
axes[2].imshow(images_flat[0].reshape(28, 28), cmap='gray', vmin=0, vmax=1)
axes[2].set_title(f"flattened: (784,) → display as (28,28)")
axes[2].axis('off')

fig.suptitle("Image Transform Pipeline", y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
assert images_norm.dtype == np.float32
assert images_norm.shape == images_raw.shape
assert images_norm.min() >= 0.0 and images_norm.max() <= 1.0

assert images_flat.shape == (60000, 784)
assert images_flat.dtype == np.float32
assert np.allclose(images_flat[0], images_norm[0].reshape(-1))

print("image transform validation passed")

## 3. one-hot 인코딩 확인

`load_labels`가 반환한 원본 `uint8` 레이블 배열에 `one_hot`을 적용한다.
각 정수 레이블이 길이 10인 벡터로 어떻게 변환되는지, 해당 위치에만 1.0이 설정되는지 확인한다.

In [ ]:
labels_raw = load_labels("train")   # (60000,) uint8
labels_oh = T.one_hot(labels_raw)   # (60000, 10) float32

print(f"raw    : shape={labels_raw.shape}, dtype={labels_raw.dtype}")
print(f"one_hot: shape={labels_oh.shape}, dtype={labels_oh.dtype}")
print()
print("첫 5개 레이블과 one-hot 벡터:")
for i in range(5):
    print(f"  label={labels_raw[i]}  →  {labels_oh[i]}")

In [ ]:
n_show = 15
fig, ax = plt.subplots(figsize=(10, 4))

im = ax.imshow(labels_oh[:n_show], cmap='Blues', vmin=0, vmax=1, aspect='auto')
ax.set_title("one-hot encoding (first 15 samples)")
ax.set_xlabel("class index")
ax.set_ylabel("sample index")
ax.set_xticks(range(10))
ax.set_yticks(range(n_show))
ax.set_yticklabels([f"{i} (label={labels_raw[i]})" for i in range(n_show)])
fig.colorbar(im, ax=ax)
fig.tight_layout()
plt.show()

In [ ]:
assert labels_oh.shape == (60000, 10)
assert labels_oh.dtype == np.float32
assert np.all(labels_oh.sum(axis=1) == 1.0), "row sum != 1"
assert set(np.unique(labels_oh)).issubset({0.0, 1.0}), "unexpected values"

# label 위치 검증
for i in range(10):
    assert labels_oh[i, labels_raw[i]] == 1.0, f"wrong position at sample {i}"

print("one_hot validation passed")

## 4. MulticlassDataset 생성 및 확인

`MulticlassDataset`은 인자 없이 `split`만 지정하면 `normalize + to_flat`과 `one_hot`을 기본값으로 적용한다.
생성자에서 모든 변환이 완료되므로 `__getitem__` 호출 시 추가 연산 없이 배열 인덱싱만 수행한다.

In [ ]:
train_ds = MulticlassDataset("train")
test_ds  = MulticlassDataset("test")

print(f"train: len={len(train_ds)}")
print(f"  images.shape={train_ds.images.shape}, images.dtype={train_ds.images.dtype}")
print(f"  targets.shape={train_ds.targets.shape}, targets.dtype={train_ds.targets.dtype}")
print()
print(f"test: len={len(test_ds)}")
print(f"  images.shape={test_ds.images.shape}, images.dtype={test_ds.images.dtype}")
print(f"  targets.shape={test_ds.targets.shape}, targets.dtype={test_ds.targets.dtype}")

img, tgt = train_ds[0]
print(f"\ntrain_ds[0]: image.shape={img.shape}, target.shape={tgt.shape}")
print(f"  image[:5] = {img[:5]}")
print(f"  target    = {tgt}")

In [ ]:
assert len(train_ds) == 60000
assert len(test_ds) == 10000

assert train_ds.images.shape == (60000, 784)
assert train_ds.images.dtype == np.float32
assert train_ds.images.min() >= 0.0 and train_ds.images.max() <= 1.0

assert train_ds.targets.shape == (60000, 10)
assert train_ds.targets.dtype == np.float32
assert np.all(train_ds.targets.sum(axis=1) == 1.0)

img, tgt = train_ds[0]
assert img.shape == (784,)
assert tgt.shape == (10,)

print("MulticlassDataset validation passed")

## 5. Dataloader 배치 생성

`Dataloader`는 `MulticlassDataset`을 받아 `batch_size` 단위의 이미지 배치와 target 배치를 yield한다.
`shuffle=True`이면 에폭마다 다른 순서로 샘플을 반환하여 학습 일반화를 높인다.

In [ ]:
import math

BATCH_SIZE = 256
train_loader = Dataloader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = Dataloader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

print(f"train_loader: len={len(train_loader)} batches  "
      f"(ceil({len(train_ds)} / {BATCH_SIZE}) = {math.ceil(len(train_ds)/BATCH_SIZE)})")
print(f"test_loader : len={len(test_loader)} batches  "
      f"(ceil({len(test_ds)} / {BATCH_SIZE}) = {math.ceil(len(test_ds)/BATCH_SIZE)})")

x_batch, y_batch = next(iter(train_loader))
print(f"\nfirst batch: x.shape={x_batch.shape}, y.shape={y_batch.shape}")
print(f"  x.dtype={x_batch.dtype}, y.dtype={y_batch.dtype}")
print(f"  x.min={x_batch.min():.4f}, x.max={x_batch.max():.4f}")
print(f"  y row sums (first 5): {y_batch[:5].sum(axis=1)}")

In [ ]:
# 학습 루프 시연 — 1 epoch, 첫 3 배치만 출력
print("학습 루프 시연 (1 epoch, 첫 3 배치):")
for batch_idx, (x, y) in enumerate(train_loader):
    if batch_idx < 3:
        print(f"  batch {batch_idx:3d}: x={x.shape}, y={y.shape}")
    elif batch_idx == 3:
        print(f"  ...")
        break

# 마지막 배치 크기 확인
batches = list(train_loader)
x_last, y_last = batches[-1]
expected_last = len(train_ds) % BATCH_SIZE
print(f"\nlast batch: x={x_last.shape}  (60000 mod {BATCH_SIZE} = {expected_last})")

In [ ]:
assert len(train_loader) == math.ceil(len(train_ds) / BATCH_SIZE)
assert sum(len(x) for x, _ in train_loader) == len(train_ds)

x0, y0 = next(iter(train_loader))
assert x0.shape == (BATCH_SIZE, 784)
assert y0.shape == (BATCH_SIZE, 10)
assert x0.dtype == np.float32
assert y0.dtype == np.float32
assert np.all(y0.sum(axis=1) == 1.0), "one-hot row sum != 1 in batch"

print("Dataloader validation passed")

## 6. 샘플 이미지 시각화

train Dataset에서 처음 10개 샘플을 꺼내 이미지와 예측 클래스를 표시한다.
`__getitem__`이 반환하는 `(image, target)` tuple을 직접 사용하는 방식이다.

In [ ]:
n_show = 10
fig, axes = plt.subplots(1, n_show, figsize=(14, 2))

for i in range(n_show):
    img, tgt = train_ds[i]
    label = np.argmax(tgt)           # one-hot → class index
    axes[i].imshow(img.reshape(28, 28), cmap='gray', vmin=0, vmax=1)
    axes[i].set_title(f"label={label}")
    axes[i].axis('off')

fig.suptitle("MulticlassDataset — first 10 samples", y=1.05)
fig.tight_layout()
plt.show()

## 요약

이 노트북에서 확인한 내용은 다음과 같다.

| 항목 | 입력 | 출력 | 비고 |
|---|---|---|---|
| `normalize` | `(N, 28, 28) uint8` | `(N, 28, 28) float32` | `/255.0` |
| `to_flat` | `(N, 28, 28) float32` | `(N, 784) float32` | `reshape` |
| `one_hot` | `(N,) uint8` | `(N, 10) float32` | 해당 위치 1.0 |
| `MulticlassDataset` | `split` | `images (N,784)`, `targets (N,10)` | eager 변환 |
| `Dataloader` | `MulticlassDataset` | `(x, y)` batch tuple | `ceil(N/B)` 배치 |

**핵심 설계 원칙**
- transform은 생성자에서 전체 배열에 한 번만 적용(eager)하므로 `__getitem__`은 배열 인덱싱만 수행한다.
- `MulticlassDataset`은 기본 transform을 내장하여 인자 없이 `split`만 지정하면 바로 사용할 수 있다.
- `for x, y in loader` 한 줄로 mini-batch 학습 루프를 구성할 수 있다.